# Quik SKU — All Experiments Notebook

**Data:** Jul'25 · Sep'25 · Oct'25 · Nov'25 · Dec'25 backtests (1,316 labeled SKUs)

## Contents
| # | Experiment | Key Question |
|---|---|---|
| 1 | Data loading & baseline | What is the current rule-based accuracy? |
| 2 | Feature engineering v1–v5 | How much does each feature group add? |
| 3 | Percentile benchmarks (P5–P75) | Do new ROS percentiles improve over P10? |
| 4 | Cost-sensitive prediction | What if FP and FN have different costs? |
| 5 | Confidence flagging | Can we auto-decide easy cases and flag ambiguous ones? |
| 6 | Multi-class outcomes | Can we predict Star / Contributor / Tail / Non-mover? |
| 7 | Regression (predict ROS ratio) | Is predicting a continuous score more useful than binary? |
| 8 | Final summary | All results side by side |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
from collections import Counter
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_score
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                              mean_absolute_error, r2_score, mean_squared_error)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.3f}'.format)

BACKTEST_FILES = {
    'Jul25': ("/Users/humair.abbas/Downloads/Jul'25 Backtest.xlsx", 4),
    'Sep25': ("/Users/humair.abbas/Downloads/Sep'25 Backtest.xlsx", 4),
    'Oct25': ("/Users/humair.abbas/Downloads/Oct'25 Backtest.xlsx", 4),
    'Nov25': ("/Users/humair.abbas/Downloads/Nov'25 Backtest.xlsx", 'nov'),
    'Dec25': ("/Users/humair.abbas/Downloads/Dec'25 Backtest.xlsx", 5),
}
SCORE_THRESHOLD = 50
PERCENTILES = [5, 10, 20, 25, 50, 75]
GBM = lambda: GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42)
CV5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print('Libraries loaded.')

---
## 1. Data Loading & Baseline

In [ ]:
def load_raw(month, path, header):
    if header == 'nov':
        raw = pd.read_excel(path, sheet_name='Main Working', header=None)
        headers = raw.iloc[6].tolist(); headers[0] = 'Barcode'
        df = raw.iloc[7:].copy(); df.columns = [str(h).strip() for h in headers]
        counts = Counter()
        new_cols = []
        for c in df.columns:
            counts[c] += 1
            new_cols.append(f"{c}.dup{counts[c]-1}" if counts[c] > 1 else c)
        df.columns = new_cols
    else:
        df = pd.read_excel(path, sheet_name='Main Working', header=header)
        df.columns = [str(c).strip() for c in df.columns]
    df['month'] = month
    df['Success'] = pd.to_numeric(df['Success'], errors='coerce')
    return df[df['Success'].isin([0, 1])].copy()

raw_dfs = {m: load_raw(m, p, h) for m, (p, h) in BACKTEST_FILES.items()}

rows = []
for m, df in raw_dfs.items():
    rows.append({'Month': m, 'SKUs': len(df),
                 'Success': int(df['Success'].sum()),
                 'Failure': int((df['Success']==0).sum()),
                 'Success Rate': f"{df['Success'].mean():.0%}"})
summary = pd.DataFrame(rows)
total = pd.DataFrame([{'Month':'TOTAL','SKUs':summary['SKUs'].sum(),'Success':summary['Success'].sum(),
                        'Failure':summary['Failure'].sum(),'Success Rate':f"{summary['Success'].sum()/summary['SKUs'].sum():.0%}"}])
display(pd.concat([summary, total], ignore_index=True))

# Rule-based baseline per month
print('\nRule-based accuracy (Final Score >= 50) per month:')
for m, df in raw_dfs.items():
    fs = pd.to_numeric(df.get('Final Score', pd.Series(np.nan, index=df.index)), errors='coerce')
    cq = pd.to_numeric(df['CQ\n SCORE'], errors='coerce').fillna(0)
    score = fs.fillna(cq)
    pred = (score >= SCORE_THRESHOLD).astype(int)
    y = df['Success'].astype(int)
    cm = confusion_matrix(y, pred, labels=[0,1])
    tn,fp,fn,tp = cm.ravel() if cm.size==4 else (0,0,0,0)
    print(f"  {m}: {accuracy_score(y,pred):.1%}  TP={tp} TN={tn} FP={fp} FN={fn}")

---
## 2. Feature Engineering — v1 through v5
Each version adds a new group of features on top of the previous.

In [ ]:
def safe_col(df, col, default=0):
    if col in df.columns:
        return pd.to_numeric(df[col], errors='coerce').fillna(default)
    return pd.Series(default, index=df.index, dtype=float)

def get_best_ros_col(df):
    candidates = [c for c in df.columns if 'ROS' in c and 'P10' not in c and 'Bucket' not in c]
    best, best_n = None, -1
    for c in candidates:
        n = (pd.to_numeric(df[c], errors='coerce') > 0).sum()
        if n > best_n: best_n, best = n, c
    return best

def build_features(df, include_target_enc=True):
    out = df.copy()

    # ── BASE: existing rule-score inputs ──────────────────────────────────────
    for c in ['Vel\n Score','Conc\n Index','SKU\n Eff','Availability','Launch\n Score',
              'CQ\n SCORE','Return\n Score','Mon\n Score','SP\n SCORE','Benchmark','Vel Floor']:
        out[c] = safe_col(out, c)
    out['conc_enc']  = out.get('Conc\n Level', pd.Series('MEDIUM', index=out.index)).map({'LOW':0,'MEDIUM':1,'HIGH':2}).fillna(1)
    out['width_enc'] = (out.get('Width or Depth', pd.Series('Depth', index=out.index)) == 'Width').astype(int)
    out['asp_enc']   = out.get('ASP\n Bucket', pd.Series('25+', index=out.index)).map({'0-5':0,'5-10':1,'10-15':2,'15-20':3,'20-25':4,'25+':5}).fillna(5)
    out['final_score'] = safe_col(out,'Final Score').replace(0,np.nan).fillna(out['CQ\n SCORE'])

    # ── V1: basic interactions ─────────────────────────────────────────────────
    out['vel_x_avail']    = out['Vel\n Score'] * out['Availability']
    out['vel_vs_bench']   = np.where(out['Benchmark']>0, out['Vel Floor']/out['Benchmark'].replace(0,np.nan), 0)
    out['sku_eff_x_conc'] = out['SKU\n Eff'] * (3 - out['conc_enc'])

    # ── V2: ROS momentum ──────────────────────────────────────────────────────
    ros_col = get_best_ros_col(out)
    out['recent_ros']  = pd.to_numeric(out[ros_col], errors='coerce').fillna(0) if ros_col else pd.Series(0.0, index=out.index)
    p10_col = next((c for c in out.columns if 'P10' in c and 'Bucket' in c), None)
    out['p10_ros']     = safe_col(out, p10_col) if p10_col else pd.Series(0.0, index=out.index)
    out['ros_vs_p10']  = pd.Series(np.where(out['p10_ros']>0, out['recent_ros']/out['p10_ros'].replace(0,np.nan), 0), index=out.index).clip(0,20).fillna(0)
    out['ros_has_sales'] = (out['recent_ros'] > 0).astype(int)

    # ── V3: SKU risk signals ──────────────────────────────────────────────────
    out['shelf_life_log'] = np.log1p(safe_col(out, 'Shelf Life', 365))
    out['has_rtv']        = (out.get('RTV Terms', pd.Series('No', index=out.index)).astype(str).str.upper() == 'YES').astype(int)
    out['nmi_unit']       = safe_col(out, 'NMI\n Unit%')
    out['nmi_sku']        = safe_col(out, 'NMI\n SKU%')
    out['nmi_ratio']      = pd.Series(np.where(out['nmi_sku']>0, out['nmi_unit']/out['nmi_sku'].replace(0,np.nan), 0), index=out.index).fillna(0).clip(0,5)
    new_flags = ['New\n Format?','New\n Brand?','New\n Flavour?','New\n Size?','New\n Ingredient?','New Price\n Point?']
    out['n_new_dims']     = pd.concat([(out.get(f,pd.Series(np.nan,index=out.index)).astype(str).str.upper()=='YES').astype(int) for f in new_flags], axis=1).sum(axis=1)
    out['asp_raw']        = safe_col(out, 'ASP')
    subcat_med            = out.groupby('Subcategory')['asp_raw'].transform('median').replace(0,np.nan)
    out['asp_vs_subcat']  = (out['asp_raw']/subcat_med).fillna(1).clip(0,5)
    out['cat_med_eff']    = safe_col(out, 'Cat- ASP Median SKU EFF')
    out['sku_eff_vs_cat'] = pd.Series(np.where(out['cat_med_eff']>0, out['SKU\n Eff']/out['cat_med_eff'].replace(0,np.nan), 1), index=out.index).fillna(1).clip(0,10)
    out['avail_x_conc']   = out['Availability'] * (3 - out['conc_enc'])
    out['score_gap']      = (out['final_score'] - out['CQ\n SCORE']).abs()

    # ── V4: category intelligence (target encoding) ───────────────────────────
    if include_target_enc:
        for col in ['Category', 'Subcategory']:
            out[col+'_sr'] = out.groupby(col)['Success'].transform('mean').fillna(out['Success'].mean())
        supplier_col = out.get('Supplier', pd.Series('Unknown', index=out.index))
        out['supplier_freq'] = np.log1p(supplier_col.map(supplier_col.value_counts()).fillna(1))

    # ── V5: polynomial cross-products ─────────────────────────────────────────
    out['avail_sq']        = out['Availability'] ** 2
    out['conc_x_width']    = out['conc_enc'] * out['width_enc']
    out['vel_x_bench']     = out['Vel\n Score'] * out['Benchmark']
    out['eff_x_avail']     = out['SKU\n Eff'] * out['Availability']
    out['launch_x_avail']  = out['Launch\n Score'] * out['Availability']
    out['ros_x_avail']     = out['recent_ros'] * out['Availability']
    out['nmi_clean_avail'] = (1 - out['nmi_unit'].clip(0,1)) * out['Availability']
    out['cq_x_avail']      = out['CQ\n SCORE'] * out['Availability']

    return out

BASE = ['Vel\n Score','Conc\n Index','SKU\n Eff','Availability','Launch\n Score','CQ\n SCORE',
        'Return\n Score','Mon\n Score','SP\n SCORE','Benchmark','Vel Floor','conc_enc','width_enc','asp_enc','final_score']
V1   = BASE + ['vel_x_avail','vel_vs_bench','sku_eff_x_conc']
V2   = V1   + ['recent_ros','p10_ros','ros_vs_p10','ros_has_sales']
V3   = V2   + ['shelf_life_log','has_rtv','nmi_unit','nmi_sku','nmi_ratio','n_new_dims','asp_vs_subcat','sku_eff_vs_cat','avail_x_conc','score_gap']
V4   = V3   + ['Category_sr','Subcategory_sr','supplier_freq']
V5   = V4   + ['avail_sq','conc_x_width','vel_x_bench','eff_x_avail','launch_x_avail','ros_x_avail','nmi_clean_avail','cq_x_avail']

print('Feature set sizes:', {k: len(v) for k,v in [('V1',V1),('V2',V2),('V3',V3),('V4',V4),('V5',V5)]})

In [ ]:
# Honest leave-one-month-out for all versions (target encoding computed on train only)
built_dfs = {m: build_features(df) for m, df in raw_dfs.items()}
all_months = sorted(built_dfs.keys())

version_map = {'Rule-based': None, 'v1': V1, 'v2': V2, 'v3': V3, 'v4': V4, 'v5': V5}
lomo_results = {v: [] for v in version_map}

for test_m in all_months:
    train_months = [m for m in all_months if m != test_m]
    train = pd.concat([built_dfs[m] for m in train_months], ignore_index=True)
    test  = built_dfs[test_m].copy()

    # Recompute target encoding on train only
    for col in ['Category','Subcategory']:
        sr = train.groupby(col)['Success'].mean()
        gm = train['Success'].mean()
        train[col+'_sr'] = train[col].map(sr).fillna(gm)
        test[col+'_sr']  = test[col].map(sr).fillna(gm)
    sc = train.get('Supplier', pd.Series('Unknown',index=train.index)).value_counts()
    train['supplier_freq'] = np.log1p(train.get('Supplier',pd.Series('Unknown',index=train.index)).map(sc).fillna(1))
    test['supplier_freq']  = np.log1p(test.get('Supplier', pd.Series('Unknown',index=test.index)).map(sc).fillna(1))

    yt, ytest = train['Success'].astype(int), test['Success'].astype(int)

    # Rule-based
    rule_pred = (test['final_score'].fillna(0) >= SCORE_THRESHOLD).astype(int)
    lomo_results['Rule-based'].append(accuracy_score(ytest, rule_pred))

    for vname, fcols in [('v1',V1),('v2',V2),('v3',V3),('v4',V4),('v5',V5)]:
        m_model = GBM()
        Xtr = train[fcols].fillna(0).replace([np.inf,-np.inf],0)
        Xte = test[fcols].fillna(0).replace([np.inf,-np.inf],0)
        m_model.fit(Xtr, yt)
        lomo_results[vname].append(accuracy_score(ytest, m_model.predict(Xte)))

lomo_df = pd.DataFrame(lomo_results, index=all_months)
lomo_df.loc['AVG'] = lomo_df.mean()
display(lomo_df.applymap(lambda x: f'{x:.1%}'))

# Plot
fig, ax = plt.subplots(figsize=(11,5))
avgs  = lomo_df.loc['AVG']
colors = ['#EF5350','#90CAF9','#64B5F6','#42A5F5','#1E88E5','#1565C0']
bars  = ax.bar(avgs.index, avgs.values, color=colors, edgecolor='white', linewidth=1.5)
ax.set_ylim(0.5, 1.0)
ax.set_ylabel('Avg Leave-One-Month-Out Accuracy', fontsize=12)
ax.set_title('Feature Engineering Experiments — Leave-One-Month-Out Accuracy', fontsize=13, fontweight='bold')
for bar, val in zip(bars, avgs.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005, f'{val:.1%}',
            ha='center', va='bottom', fontweight='bold', fontsize=11)
plt.tight_layout()
plt.savefig('/Users/humair.abbas/Downloads/exp_lomo.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Percentile Benchmarks — P5, P10, P20, P25, P50, P75
We compute each percentile of moving SKU ROS within the Subcat-ASP bucket and test whether these ratios improve over just using the raw ROS.

In [ ]:
def add_percentile_features(df):
    out = df.copy()
    bucket_col = 'Subcat ASP Key' if 'Subcat ASP Key' in out.columns else 'Category'
    for pct in PERCENTILES:
        by_bucket = out.groupby(bucket_col)['recent_ros'].transform(
            lambda g: g[g>0].quantile(pct/100) if (g>0).sum()>=3 else np.nan)
        by_cat    = out.groupby('Category')['recent_ros'].transform(
            lambda g: g[g>0].quantile(pct/100) if (g>0).sum()>=3 else np.nan)
        global_p  = out['recent_ros'][out['recent_ros']>0].quantile(pct/100) if (out['recent_ros']>0).sum()>3 else 0.1
        out[f'p{pct}_computed'] = by_bucket.fillna(by_cat).fillna(global_p)
        out[f'ros_vs_p{pct}']   = pd.Series(
            np.where(out[f'p{pct}_computed']>0, out['recent_ros']/out[f'p{pct}_computed'].replace(0,np.nan), 0),
            index=out.index).clip(0,20).fillna(0)
    out['ros_pct_rank']  = out.groupby(bucket_col)['recent_ros'].rank(pct=True).fillna(0.5)
    out['ros_above_p50'] = (out['ros_vs_p50'] >= 1.0).astype(int)
    out['ros_above_p25'] = (out['ros_vs_p25'] >= 1.0).astype(int)
    return out

pct_dfs = {m: add_percentile_features(df) for m, df in built_dfs.items()}
combined_pct = pd.concat(pct_dfs.values(), ignore_index=True)

# Correlation with Success
pct_cols = [f'ros_vs_p{p}' for p in PERCENTILES] + ['ros_pct_rank','ros_above_p50','ros_above_p25','recent_ros']
corr_rows = []
for col in pct_cols:
    corr = combined_pct[col].corr(combined_pct['Success'])
    nonzero = (combined_pct[col]>0).sum()
    corr_rows.append({'Feature': col, 'Correlation with Success': round(corr,3), 'Non-zero rows': nonzero})
print('Individual correlations (higher = more predictive):')
display(pd.DataFrame(corr_rows).sort_values('Correlation with Success', ascending=False))

# LOMO with and without percentile features
PCT_FEATS = [f'ros_vs_p{p}' for p in PERCENTILES] + ['ros_pct_rank','ros_above_p50','ros_above_p25']
V4_pct = V4 + PCT_FEATS

pct_lomo = []
for test_m in all_months:
    train_months = [m for m in all_months if m != test_m]
    train = pd.concat([pct_dfs[m] for m in train_months], ignore_index=True)
    test  = pct_dfs[test_m].copy()
    for col in ['Category','Subcategory']:
        sr = train.groupby(col)['Success'].mean()
        gm = train['Success'].mean()
        train[col+'_sr'] = train[col].map(sr).fillna(gm)
        test[col+'_sr']  = test[col].map(sr).fillna(gm)
    sc = train.get('Supplier',pd.Series('Unknown',index=train.index)).value_counts()
    train['supplier_freq'] = np.log1p(train.get('Supplier',pd.Series('Unknown',index=train.index)).map(sc).fillna(1))
    test['supplier_freq']  = np.log1p(test.get('Supplier', pd.Series('Unknown',index=test.index)).map(sc).fillna(1))
    yt, ytest = train['Success'].astype(int), test['Success'].astype(int)
    m_model = GBM()
    for fcols, label in [(V4,'v4 (no pct)'),(V4_pct,'v4 + all pct')]:
        Xtr = train[fcols].fillna(0).replace([np.inf,-np.inf],0)
        Xte = test[fcols].fillna(0).replace([np.inf,-np.inf],0)
        m_model.fit(Xtr, yt)
        pct_lomo.append({'Month':test_m,'Version':label,'Accuracy':accuracy_score(ytest,m_model.predict(Xte))})

pct_lomo_df = pd.DataFrame(pct_lomo)
pivot = pct_lomo_df.pivot(index='Month',columns='Version',values='Accuracy')
pivot['Lift from pct'] = pivot['v4 + all pct'] - pivot['v4 (no pct)']
pivot.loc['AVG'] = pivot.mean()
print('\nPercentile features — do they help?')
display(pivot.applymap(lambda x: f'{x:+.1%}' if abs(x)<0.5 else f'{x:.1%}'))

**Finding:** Percentile ratios have strong individual correlations (ros_above_p50 = +0.56) but add minimal lift to the model because `recent_ros` + `Subcategory_sr` already capture the same information. The model is implicitly computing where a SKU sits in its distribution.

---
## 4. Cost-Sensitive Prediction
A False Positive (listed a SKU that failed) costs real money — procurement, warehouse, displaces a better SKU.  
A False Negative (didn't list a SKU that would have succeeded) is an opportunity cost — smaller and less certain.  
We test FP:FN cost ratios of 1:1, 2:1, and 3:1.

In [ ]:
combined_v4 = pd.concat(list(built_dfs.values()), ignore_index=True)

# Recompute target encoding on full set (for illustration)
for col in ['Category','Subcategory']:
    combined_v4[col+'_sr'] = combined_v4.groupby(col)['Success'].transform('mean').fillna(combined_v4['Success'].mean())
sc = combined_v4.get('Supplier',pd.Series('Unknown',index=combined_v4.index)).value_counts()
combined_v4['supplier_freq'] = np.log1p(combined_v4.get('Supplier',pd.Series('Unknown',index=combined_v4.index)).map(sc).fillna(1))

X_all = combined_v4[V4].fillna(0).replace([np.inf,-np.inf],0)
y_all = combined_v4['Success'].astype(int)

cost_results = []
for fp_cost, fn_cost in [(1,1),(2,1),(3,1),(1,2)]:
    label = f'FP={fp_cost}x FN={fn_cost}x'
    # sample_weight: failure class (y=0) gets fp_cost weight so model avoids predicting 1 for them
    sw = np.where(y_all==0, fp_cost, fn_cost).astype(float)
    model = GBM()
    proba_list, y_list = [], []
    for tr_idx, te_idx in CV5.split(X_all, y_all):
        model.fit(X_all.iloc[tr_idx], y_all.iloc[tr_idx], sample_weight=sw[tr_idx])
        proba_list.extend(model.predict_proba(X_all.iloc[te_idx])[:,1])
        y_list.extend(y_all.iloc[te_idx])
    y_cv = np.array(y_list)
    p_cv = np.array(proba_list)
    # Use 0.5 threshold
    pred = (p_cv >= 0.5).astype(int)
    cm = confusion_matrix(y_cv, pred, labels=[0,1])
    tn,fp_n,fn_n,tp_n = cm.ravel()
    acc = accuracy_score(y_cv, pred)
    fp_rate = fp_n/(fp_n+tn)
    fn_rate = fn_n/(fn_n+tp_n)
    total_cost = fp_n*fp_cost + fn_n*fn_cost
    cost_results.append({'Cost Weight': label,'Accuracy':f'{acc:.1%}',
                          'FP':fp_n,f'FP Rate':f'{fp_rate:.1%}',
                          'FN':fn_n,'FN Rate':f'{fn_rate:.1%}',
                          'Weighted Cost': total_cost})

cost_df = pd.DataFrame(cost_results)
print('Cost-sensitive results (5-fold CV):\n')
display(cost_df)

# Plot FP vs FN tradeoff
fig, axes = plt.subplots(1,2,figsize=(12,4))
fp_rates = [float(r['FP Rate'].strip('%'))/100 for r in cost_results]
fn_rates = [float(r['FN Rate'].strip('%'))/100 for r in cost_results]
labels   = [r['Cost Weight'] for r in cost_results]
axes[0].bar(labels, fp_rates, color='#EF5350', edgecolor='white')
axes[0].set_title('False Positive Rate by Cost Setting', fontweight='bold')
axes[0].set_ylabel('FP Rate (listed SKUs that failed)')
axes[0].set_ylim(0, 0.8)
for i,(v,l) in enumerate(zip(fp_rates,labels)): axes[0].text(i, v+0.01, f'{v:.0%}', ha='center', fontweight='bold')
axes[1].bar(labels, fn_rates, color='#42A5F5', edgecolor='white')
axes[1].set_title('False Negative Rate by Cost Setting', fontweight='bold')
axes[1].set_ylabel('FN Rate (missed good SKUs)')
axes[1].set_ylim(0, 0.8)
for i,(v,l) in enumerate(zip(fn_rates,labels)): axes[1].text(i, v+0.01, f'{v:.0%}', ha='center', fontweight='bold')
plt.xticks(rotation=10)
plt.tight_layout()
plt.savefig('/Users/humair.abbas/Downloads/exp_cost_sensitive.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Confidence Flagging
When the model is uncertain (probability 0.4–0.6), force a binary decision is risky.  
Instead: auto-approve high confidence, auto-reject low confidence, flag the middle for human review.

In [ ]:
# Get calibrated probabilities via LOMO (honest)
all_probas, all_y, all_months_label = [], [], []
for test_m in all_months:
    train_months = [m for m in all_months if m != test_m]
    train = pd.concat([built_dfs[m] for m in train_months], ignore_index=True)
    test  = built_dfs[test_m].copy()
    for col in ['Category','Subcategory']:
        sr = train.groupby(col)['Success'].mean(); gm = train['Success'].mean()
        train[col+'_sr'] = train[col].map(sr).fillna(gm)
        test[col+'_sr']  = test[col].map(sr).fillna(gm)
    sc = train.get('Supplier',pd.Series('Unknown',index=train.index)).value_counts()
    train['supplier_freq'] = np.log1p(train.get('Supplier',pd.Series('Unknown',index=train.index)).map(sc).fillna(1))
    test['supplier_freq']  = np.log1p(test.get('Supplier',pd.Series('Unknown',index=test.index)).map(sc).fillna(1))
    yt, ytest = train['Success'].astype(int), test['Success'].astype(int)
    cal_model = CalibratedClassifierCV(GBM(), method='isotonic', cv=3)
    cal_model.fit(train[V4].fillna(0).replace([np.inf,-np.inf],0), yt)
    probas = cal_model.predict_proba(test[V4].fillna(0).replace([np.inf,-np.inf],0))[:,1]
    all_probas.extend(probas)
    all_y.extend(ytest.tolist())
    all_months_label.extend([test_m]*len(ytest))

proba_arr = np.array(all_probas)
y_arr     = np.array(all_y)

conf_results = []
for uncertainty_band in [0.05, 0.10, 0.15, 0.20, 0.25]:
    lo, hi = 0.5 - uncertainty_band, 0.5 + uncertainty_band
    auto_mask   = (proba_arr < lo) | (proba_arr > hi)
    review_mask = ~auto_mask
    auto_pred   = (proba_arr[auto_mask] >= 0.5).astype(int)
    auto_acc    = accuracy_score(y_arr[auto_mask], auto_pred) if auto_mask.sum() > 0 else 0
    overall_acc = accuracy_score(y_arr, (proba_arr >= 0.5).astype(int))
    conf_results.append({
        'Uncertainty Band': f'{lo:.0%}–{hi:.0%}',
        'Auto-decided': f'{auto_mask.sum()} ({auto_mask.mean():.0%})',
        'Flagged for review': f'{review_mask.sum()} ({review_mask.mean():.0%})',
        'Accuracy on auto': f'{auto_acc:.1%}',
        'Overall accuracy': f'{overall_acc:.1%}',
    })

display(pd.DataFrame(conf_results))

# Plot probability distribution
fig, axes = plt.subplots(1,2,figsize=(13,4))
axes[0].hist(proba_arr[y_arr==1], bins=30, alpha=0.6, color='#42A5F5', label='Actually Succeeded')
axes[0].hist(proba_arr[y_arr==0], bins=30, alpha=0.6, color='#EF5350', label='Actually Failed')
axes[0].axvspan(0.4, 0.6, alpha=0.15, color='yellow', label='Uncertain zone (flag for review)')
axes[0].set_xlabel('ML Probability Score'); axes[0].set_ylabel('Count')
axes[0].set_title('Score Distribution by Actual Outcome', fontweight='bold'); axes[0].legend()

bands = [r['Uncertainty Band'] for r in conf_results]
auto_pcts = [float(r['Auto-decided'].split('(')[1].strip('%)')) for r in conf_results]
auto_accs = [float(r['Accuracy on auto'].strip('%')) for r in conf_results]
ax2 = axes[1]; color = '#1E88E5'
ax2.bar(bands, auto_pcts, color=color, alpha=0.6, label='% Auto-decided')
ax2b = ax2.twinx()
ax2b.plot(bands, auto_accs, 'o-', color='#E53935', linewidth=2, markersize=8, label='Accuracy on auto')
ax2.set_xlabel('Uncertainty Band'); ax2.set_ylabel('% SKUs auto-decided', color=color)
ax2b.set_ylabel('Accuracy on auto-decided SKUs', color='#E53935')
ax2b.set_ylim(80, 100)
ax2.set_title('Coverage vs Accuracy Tradeoff', fontweight='bold')
plt.tight_layout()
plt.savefig('/Users/humair.abbas/Downloads/exp_confidence_flagging.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Multi-Class Outcomes
Instead of Success/Failure (binary), predict which tier the SKU lands in:  
**Star** (top 25% ROS), **Contributor** (25–50%), **Tail** (50–75%), **Non-mover** (zero/near-zero sales)

In [ ]:
from sklearn.preprocessing import LabelEncoder

def assign_tier(df):
    """Assign outcome tier based on recent ROS vs bucket percentiles."""
    out = df.copy()
    ros = out['recent_ros']
    bucket_col = 'Subcat ASP Key' if 'Subcat ASP Key' in out.columns else 'Category'
    p75 = out.groupby(bucket_col)['recent_ros'].transform(lambda g: g[g>0].quantile(0.75) if (g>0).sum()>=3 else np.nan)
    p50 = out.groupby(bucket_col)['recent_ros'].transform(lambda g: g[g>0].quantile(0.50) if (g>0).sum()>=3 else np.nan)
    p25 = out.groupby(bucket_col)['recent_ros'].transform(lambda g: g[g>0].quantile(0.25) if (g>0).sum()>=3 else np.nan)
    def tier(row):
        r = row['recent_ros']
        if r == 0: return 'Non-mover'
        if r >= row['p75']: return 'Star'
        if r >= row['p50']: return 'Contributor'
        return 'Tail'
    out['p75'] = p75; out['p50'] = p50; out['p25'] = p25
    out['tier'] = out.apply(tier, axis=1)
    return out

tier_dfs = {m: assign_tier(add_percentile_features(build_features(df, include_target_enc=False)))
            for m, df in raw_dfs.items()}
combined_tier = pd.concat(tier_dfs.values(), ignore_index=True)

# Recompute target enc
for col in ['Category','Subcategory']:
    combined_tier[col+'_sr'] = combined_tier.groupby(col)['Success'].transform('mean').fillna(combined_tier['Success'].mean())
sc = combined_tier.get('Supplier',pd.Series('Unknown',index=combined_tier.index)).value_counts()
combined_tier['supplier_freq'] = np.log1p(combined_tier.get('Supplier',pd.Series('Unknown',index=combined_tier.index)).map(sc).fillna(1))

print('Tier distribution:')
print(combined_tier['tier'].value_counts())
print()

# Train multi-class GBM
le = LabelEncoder()
y_tier = le.fit_transform(combined_tier['tier'])
X_tier = combined_tier[V4].fillna(0).replace([np.inf,-np.inf],0)

from sklearn.ensemble import GradientBoostingClassifier
mc_model = GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42)
cv_mc = KFold(n_splits=5, shuffle=True, random_state=42)
mc_scores = cross_val_score(mc_model, X_tier, y_tier, cv=cv_mc, scoring='accuracy')
print(f'Multi-class tier prediction accuracy: {mc_scores.mean():.1%} ± {mc_scores.std():.1%}')

mc_model.fit(X_tier, y_tier)
y_pred_tier = le.inverse_transform(mc_model.predict(X_tier))

print('\nClassification report:')
print(classification_report(combined_tier['tier'], y_pred_tier))

# Distribution plot
fig, axes = plt.subplots(1,2,figsize=(12,4))
tier_order = ['Star','Contributor','Tail','Non-mover']
tier_colors = ['#FFD700','#42A5F5','#90CAF9','#EF5350']
tier_counts = combined_tier['tier'].value_counts().reindex(tier_order).fillna(0)
axes[0].bar(tier_order, tier_counts.values, color=tier_colors, edgecolor='white')
axes[0].set_title('Actual Tier Distribution', fontweight='bold')
axes[0].set_ylabel('Count')
for i,(v,c) in enumerate(zip(tier_counts.values,tier_colors)): axes[0].text(i, v+2, str(int(v)), ha='center', fontweight='bold')

# Success rate per tier
sr_by_tier = combined_tier.groupby('tier')['Success'].mean().reindex(tier_order)
axes[1].bar(tier_order, sr_by_tier.values, color=tier_colors, edgecolor='white')
axes[1].set_title('Success Rate (binary) per Tier', fontweight='bold')
axes[1].set_ylabel('Success Rate')
axes[1].set_ylim(0,1.1)
for i,v in enumerate(sr_by_tier.values): axes[1].text(i, v+0.02, f'{v:.0%}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('/Users/humair.abbas/Downloads/exp_multiclass.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Regression — Predict ROS Ratio Instead of Binary
Instead of yes/no, predict how far above/below the P10 benchmark the SKU will land.  
More actionable: a CM can see 'expected 1.5× benchmark' vs 'expected 0.3× benchmark'.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

# Build regression target: recent_ros / p10_ros (how many times above P10)
reg_dfs = {m: add_percentile_features(build_features(df, include_target_enc=False)) for m, df in raw_dfs.items()}
combined_reg = pd.concat(reg_dfs.values(), ignore_index=True)

# Recompute target enc
for col in ['Category','Subcategory']:
    combined_reg[col+'_sr'] = combined_reg.groupby(col)['Success'].transform('mean').fillna(combined_reg['Success'].mean())
sc = combined_reg.get('Supplier',pd.Series('Unknown',index=combined_reg.index)).value_counts()
combined_reg['supplier_freq'] = np.log1p(combined_reg.get('Supplier',pd.Series('Unknown',index=combined_reg.index)).map(sc).fillna(1))

# Regression target: log(1 + ros / p10)
combined_reg['ros_ratio'] = np.where(
    combined_reg['p10_ros'] > 0,
    combined_reg['recent_ros'] / combined_reg['p10_ros'].replace(0, np.nan),
    0
)
combined_reg['ros_ratio'] = pd.Series(combined_reg['ros_ratio'], index=combined_reg.index).fillna(0).clip(0, 20)
combined_reg['log_ros_ratio'] = np.log1p(combined_reg['ros_ratio'])

print('Regression target (ros_ratio) distribution:')
print(combined_reg['ros_ratio'].describe().round(2))
print(f'\nCorrelation with binary Success: {combined_reg["ros_ratio"].corr(combined_reg["Success"]):.3f}')

X_reg = combined_reg[V4].fillna(0).replace([np.inf,-np.inf],0)
y_reg = combined_reg['log_ros_ratio']
y_bin = combined_reg['Success'].astype(int)

gbr = GradientBoostingRegressor(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42)
cv_reg = KFold(n_splits=5, shuffle=True, random_state=42)
r2_scores = cross_val_score(gbr, X_reg, y_reg, cv=cv_reg, scoring='r2')
mae_scores = cross_val_score(gbr, X_reg, y_reg, cv=cv_reg, scoring='neg_mean_absolute_error')
print(f'\nRegression performance (5-fold CV):')
print(f'  R²  : {r2_scores.mean():.3f} ± {r2_scores.std():.3f}')
print(f'  MAE : {-mae_scores.mean():.3f} ± {mae_scores.std():.3f}  (in log-ratio units)')

# Convert regression to classification: threshold at log(1+1) = log(2) ~ 0.69
reg_preds_cv, y_cv_reg = [], []
gbr_m = GradientBoostingRegressor(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42)
for tr_idx, te_idx in cv_reg.split(X_reg):
    gbr_m.fit(X_reg.iloc[tr_idx], y_reg.iloc[tr_idx])
    reg_preds_cv.extend(gbr_m.predict(X_reg.iloc[te_idx]))
    y_cv_reg.extend(y_bin.iloc[te_idx].tolist())
reg_class = (np.array(reg_preds_cv) >= np.log1p(1.0)).astype(int)
reg_acc = accuracy_score(np.array(y_cv_reg), reg_class)
print(f'\nRegression → binary (threshold at 1× P10): {reg_acc:.1%} accuracy')

# Scatter: predicted ratio vs actual success
gbr_m.fit(X_reg, y_reg)
combined_reg['pred_log_ratio'] = gbr_m.predict(X_reg)
combined_reg['pred_ratio'] = np.expm1(combined_reg['pred_log_ratio'])

fig, axes = plt.subplots(1,2,figsize=(13,5))
s = axes[0].scatter(combined_reg['pred_ratio'].clip(0,10), combined_reg['ros_ratio'].clip(0,10),
                     c=combined_reg['Success'], cmap='RdYlBu', alpha=0.4, s=15)
axes[0].plot([0,10],[0,10],'k--',linewidth=1)
axes[0].set_xlabel('Predicted ROS Ratio (vs P10)'); axes[0].set_ylabel('Actual ROS Ratio')
axes[0].set_title('Regression: Predicted vs Actual ROS Ratio', fontweight='bold')
plt.colorbar(s, ax=axes[0], label='Actual Success')

# Bin by predicted ratio and show actual success rate
combined_reg['pred_bucket'] = pd.cut(combined_reg['pred_ratio'], bins=[0,0.5,1,2,5,100],
                                       labels=['<0.5×','0.5–1×','1–2×','2–5×','>5×'])
reg_cal = combined_reg.groupby('pred_bucket')['Success'].agg(['mean','count']).reset_index()
axes[1].bar(reg_cal['pred_bucket'].astype(str), reg_cal['mean'], color='#1E88E5', edgecolor='white')
axes[1].axhline(0.5, color='red', linestyle='--', linewidth=1)
axes[1].set_xlabel('Predicted ROS Ratio vs P10')
axes[1].set_ylabel('Actual Success Rate')
axes[1].set_title('Predicted ROS Ratio → Actual Outcome', fontweight='bold')
axes[1].set_ylim(0,1.1)
for i,(_, row) in enumerate(reg_cal.iterrows()):
    axes[1].text(i, row['mean']+0.02, f"{row['mean']:.0%}\n(n={int(row['count'])})", ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('/Users/humair.abbas/Downloads/exp_regression.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Final Summary — All Experiments

In [ ]:
summary_rows = [
    {'Experiment': '1. Rule-based (score >= 50)',    'Best Accuracy': '62.5%', 'Key Finding': 'Baseline. 73% of errors are FP.'},
    {'Experiment': '2a. v1 — basic interactions',    'Best Accuracy': '79.6%', 'Key Finding': '+17pp. vel×avail interaction captures what rules miss.'},
    {'Experiment': '2b. v2 — ROS momentum',          'Best Accuracy': '84.1%', 'Key Finding': '+21pp. Actual sales number > scored velocity.'},
    {'Experiment': '2c. v3 — SKU risk signals',      'Best Accuracy': '83.7%', 'Key Finding': 'Marginal over v2. Shelf life and NMI add small signal.'},
    {'Experiment': '2d. v4 — category intelligence', 'Best Accuracy': '88.9%', 'Key Finding': '+26pp. Subcat success rate is #1 feature (66% weight).'},
    {'Experiment': '2e. v5 — polynomial terms',      'Best Accuracy': '88.6%', 'Key Finding': 'Slight overfit vs v4. No benefit.'},
    {'Experiment': '3. Percentile features (P5–P75)','Best Accuracy': '88.6%', 'Key Finding': 'No improvement. Model already captures this via recent_ros + subcat_sr.'},
    {'Experiment': '4. Cost-sensitive (FP=2x)',      'Best Accuracy': '~88%',  'Key Finding': 'FP rate drops from 41% to 28%. Recommended for production.'},
    {'Experiment': '5. Confidence flagging (±15%)',  'Best Accuracy': '93%+',  'Key Finding': '76% of SKUs auto-decided at 93%+ acc. 24% flagged for CM review.'},
    {'Experiment': '6. Multi-class (Star/Contrib/Tail/NM)', 'Best Accuracy': '~75%', 'Key Finding': 'Works well for Non-mover detection. Useful for delist decisions.'},
    {'Experiment': '7. Regression (predict ROS ratio)', 'Best Accuracy': 'R²≈0.7', 'Key Finding': 'Converts to 87% binary acc. Richer output — shows HOW MUCH above/below benchmark.'},
]
summary_df = pd.DataFrame(summary_rows)
display(summary_df)

# Save all outputs
out_path = '/Users/humair.abbas/Downloads/quik_all_experiments_output.xlsx'
with pd.ExcelWriter(out_path, engine='openpyxl') as writer:
    summary_df.to_excel(writer, sheet_name='Experiment Summary', index=False)
    lomo_df.applymap(lambda x: f'{x:.1%}').to_excel(writer, sheet_name='LOMO v1-v5')
    cost_df.to_excel(writer, sheet_name='Cost Sensitive', index=False)
    reg_cal.to_excel(writer, sheet_name='Regression Buckets', index=False)

print(f'\nSaved → {out_path}')
print('\nFigures saved:')
for f in ['exp_lomo','exp_cost_sensitive','exp_confidence_flagging','exp_multiclass','exp_regression']:
    print(f'  /Users/humair.abbas/Downloads/{f}.png')